In [1]:
import os, sys, importlib, csv, time, traceback
from pathlib import Path
from datetime import datetime
from collections import Counter

import pandas as pd

os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGCHAIN_TRACING"]    = "false"

# ── مسیرها ────────────────────────────────────────────────
MA_ROOT      = Path(r"F:\Thesis\project\3-Multi-Agent-System\Test_402\MA")
PROJECT_ROOT = MA_ROOT / "legal_multi_agent"
NOTEBOOK_DIR = Path(r"F:\Thesis\project\3-Multi-Agent-System\Test_402\MA")

for p in [str(MA_ROOT), str(PROJECT_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

print(f"✓ MA_ROOT         : {MA_ROOT}")
print(f"✓ PROJECT_ROOT    : {PROJECT_ROOT}")
print(f"✓ agents exists   : {(PROJECT_ROOT / 'agents').exists()}")
print(f"✓ graphs exists   : {(PROJECT_ROOT / 'graphs').exists()}")
print(f"✓ OPENROUTER_API_KEY: {(os.getenv('OPENROUTER_API_KEY') or '')[:15]} ...")
print(f"✓ COHERE_API_KEY    : {(os.getenv('COHERE_API_KEY')     or '')[:15]} ...")
print(f"✓ MODEL             : {os.getenv('MODEL', '⚠️ تعریف نشده!')}")

✓ MA_ROOT         : F:\Thesis\project\3-Multi-Agent-System\Test_402\MA
✓ PROJECT_ROOT    : F:\Thesis\project\3-Multi-Agent-System\Test_402\MA\legal_multi_agent
✓ agents exists   : True
✓ graphs exists   : True
✓ OPENROUTER_API_KEY: sk-or-v1-4bcfd4 ...
✓ COHERE_API_KEY    : O0pIM8Zkm973sO7 ...
✓ MODEL             : google/gemini-3-flash-preview


In [2]:
modules_to_reload = [
    'legal_multi_agent.utils.toon',
    'legal_multi_agent.state.schemas',
    'legal_multi_agent.agents.supervisor',
    'legal_multi_agent.agents.researcher',
    'legal_multi_agent.agents.reasoner',
    'legal_multi_agent.agents.critic',
    'legal_multi_agent.agents.tool_executor',
    'legal_multi_agent.graphs.main_graph',
]

print("🔄 Reloading modules...")
for module_name in modules_to_reload:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])
        print(f"  ♻️  Reloaded : {module_name}")
    else:
        print(f"  ⬇️  Loading  : {module_name}")

print("✅ All modules reloaded\n")

from legal_multi_agent.graphs.main_graph import graph
print("✅ Graph imported")

🔄 Reloading modules...
  ⬇️  Loading  : legal_multi_agent.utils.toon
  ⬇️  Loading  : legal_multi_agent.state.schemas
  ⬇️  Loading  : legal_multi_agent.agents.supervisor
  ⬇️  Loading  : legal_multi_agent.agents.researcher
  ⬇️  Loading  : legal_multi_agent.agents.reasoner
  ⬇️  Loading  : legal_multi_agent.agents.critic
  ⬇️  Loading  : legal_multi_agent.agents.tool_executor
  ⬇️  Loading  : legal_multi_agent.graphs.main_graph
✅ All modules reloaded

✅ Graph imported


In [4]:
# ── مسیر فایل‌های CSV ─────────────────────────────────────
QUESTIONS_CSV = r"F:\Thesis\project\403-vekalat\structured_questions.csv"
GOLD_CSV      = r"F:\Thesis\project\1-BaselineTest\GOLD\Gold.csv"

df_questions = pd.read_csv(QUESTIONS_CSV, encoding="utf-8-sig")
df_gold      = pd.read_csv(GOLD_CSV,      encoding="utf-8-sig")

# ── حذف سؤال ۸۹ ───────────────────────────────────────────
if "idx" in df_gold.columns and 89 in df_gold["idx"].values:
    df_gold = df_gold[df_gold["idx"] != 89].reset_index(drop=True)

# ── سؤالات چند جوابه ──────────────────────────────────────
multi_gold = {
    90: ["3", "4"],
    53: ["1", "2"],
    52: ["2", "4"],
    48: ["1", "3"],
    46: ["1", "3"],
    36: ["1", "4"],
    30: ["1", "2"],
    25: ["3", "4"],
    9:  ["1", "2"],
    4:  ["1", "4"],
    3:  ["2", "4"],
}

def normalize_gold_value(idx: int, gold_val: str) -> str:
    if idx in multi_gold:
        return ",".join(multi_gold[idx])
    val    = str(gold_val).strip()
    digits = [ch for ch in val if ch.isdigit()]
    if not digits:
        return val
    if len(digits) == 1:
        return digits[0]
    return ",".join(digits)

df_gold["Gold"] = [
    normalize_gold_value(int(idx), gold)
    for idx, gold in zip(df_gold["idx"], df_gold["Gold"])
]

# ── یکپارچه‌سازی ──────────────────────────────────────────
df_gold = df_gold.rename(columns={"idx": "question_number"})
df = df_questions.merge(df_gold[["question_number", "Gold"]],
                        on="question_number", how="left")

print(f"✅ سؤالات بارگذاری شد: {len(df)} سؤال")
print(f"   ستون‌ها: {list(df.columns)}")
print(f"   سؤالات چند جوابه: {len(multi_gold)} مورد → {sorted(multi_gold.keys())}")
print(f"   سؤال ۸۹ حذف شد: {89 not in df['question_number'].values}")
print("\n📋 نمونه داده:")
print(df[["question_number", "Gold", "question"]].head(3).to_string(index=False))

✅ سؤالات بارگذاری شد: 120 سؤال
   ستون‌ها: ['question_number', 'category', 'question', 'options', 'Gold']
   سؤالات چند جوابه: 11 مورد → [3, 4, 9, 25, 30, 36, 46, 48, 52, 53, 90]
   سؤال ۸۹ حذف شد: False

📋 نمونه داده:
 question_number Gold                                                                                                                                                                                                                       question
               1    3                                                                                                                                                                                                 کدام یک از موارد زیر صحیح است؟
               2    2 شخص «الف» حین رانندگی با خودرو سواری با شخص «ب» (موتورسوار) تصادف می کند شخص «الف» که دارای پوشش بیمه اجباری است صد درصد مقصر حادثه تشخیص داده می شود. بیمهگر مسئولیت، تکلیف به جبران کدام دسته از خسارات و تا چه میزانی دارد؟
               3  2,4                         

In [5]:
# ── ستون‌های مکالمه (برای هر sheet) ──────────────────────
CONV_COLUMNS = [
    "step", "node", "agent", "role", "content_type",
    "content", "meta_answer", "meta_needs_revision",
    "meta_issue", "meta_action", "meta_reasoning",
]

def make_add_row(rows_list):
    """یک تابع add_row برای یک لیست rows مستقل می‌سازد"""
    def add_row(step, node, agent, role, content_type, content,
                meta_answer=None, meta_needs_revision=None,
                meta_issue=None, meta_action=None, meta_reasoning=None):
        rows_list.append({
            "step":               step,
            "node":               node,
            "agent":              agent,
            "role":               role,
            "content_type":       content_type,
            "content":            str(content)            if content            is not None else "",
            "meta_answer":        str(meta_answer)        if meta_answer        is not None else "",
            "meta_needs_revision":str(meta_needs_revision)if meta_needs_revision is not None else "",
            "meta_issue":         str(meta_issue)         if meta_issue         is not None else "",
            "meta_action":        str(meta_action)        if meta_action        is not None else "",
            "meta_reasoning":     str(meta_reasoning)     if meta_reasoning     is not None else "",
        })
    return add_row


def run_one_question(q_num, question, options_text,
                     use_option_verifier=True,
                     use_retriever_tool=False,
                     max_revisions=2):
    """
    یک سؤال را اجرا می‌کند و dict نتیجه + لیست rows مکالمه را برمی‌گرداند.
    """
    conv_rows = []
    add_row   = make_add_row(conv_rows)

    initial_state = {
        "question_number":     q_num,
        "question":            question,
        "options_text":        options_text,
        "max_revisions":       max_revisions,
        "use_option_verifier": use_option_verifier,
        "use_retriever_tool":  use_retriever_tool,
    }

    full_state          = dict(initial_state)
    prev_messages_count = 0
    prev_draft_toon     = None
    prev_critic_toon    = None
    prev_final_toon     = None
    step_times          = {}
    t_global_start      = time.time()

    print(f"\n{'═'*65}")
    print(f"🚀 سؤال {q_num:>2}  |  {question[:55]}...")
    print(f"{'═'*65}")

    try:
        for step_num, state_snapshot in enumerate(
            graph.stream(initial_state, {"recursion_limit": 60}), start=1
        ):
            node_name = list(state_snapshot.keys())[0]
            delta     = state_snapshot[node_name]
            t_elapsed = round(time.time() - t_global_start, 2)

            for k, v in delta.items():
                if k == "messages" and isinstance(v, list):
                    existing = full_state.get("messages") or []
                    full_state["messages"] = existing + [m for m in v if m not in existing]
                else:
                    full_state[k] = v

            messages     = full_state.get("messages") or []
            new_messages = messages[prev_messages_count:]
            prev_messages_count = len(messages)

            current_draft  = full_state.get("draft_toon")
            current_critic = full_state.get("critic_toon")
            current_final  = full_state.get("final_toon")

            new_draft  = current_draft  if current_draft  != prev_draft_toon  else None
            new_critic = current_critic if current_critic != prev_critic_toon else None
            new_final  = current_final  if current_final  != prev_final_toon  else None

            prev_draft_toon  = current_draft
            prev_critic_toon = current_critic
            prev_final_toon  = current_final
            step_times[f"step_{step_num}_{node_name}"] = t_elapsed

            # ── CSV: state summary ─────────────────────────
            add_row(step_num, node_name.upper(), node_name, "system", "state_summary",
                    f"next={full_state.get('next')} | ctx={len(full_state.get('context') or '')}chars"
                    f" | n_docs={len(full_state.get('rag_results') or [])} | msgs={len(messages)}"
                    f" | elapsed={t_elapsed}s")

            # ── چاپ خلاصه step ────────────────────────────
            print(f"  📍 Step {step_num:>2} [{node_name.upper():<12}] "
                  f"next={str(full_state.get('next') or ''):<12} "
                  f"msgs={len(messages):>2} | {t_elapsed}s")

            # ── پیام‌های جدید ──────────────────────────────
            for msg in new_messages:
                if not isinstance(msg, dict):
                    continue
                meta    = msg.get("metadata") or {}
                agent   = meta.get("agent") or msg.get("name") or msg.get("role") or "?"
                role    = msg.get("role", "?")
                content = msg.get("content") or ""
                add_row(step_num, node_name.upper(), agent.upper(), role,
                        "message", content)

            # ── RAG docs ───────────────────────────────────
            if node_name.upper() == "RESEARCHER":
                for i, doc in enumerate(full_state.get("rag_results") or [], 1):
                    if isinstance(doc, dict):
                        meta_d = doc.get("metadata") or {}
                        title  = meta_d.get("title") or meta_d.get("source") or f"سند {i}"
                        add_row(step_num, "RESEARCHER", "RAG", "system",
                                f"doc_{i}_content", doc.get("content") or doc.get("page_content") or "")

            # ── Draft جدید ─────────────────────────────────
            if new_draft:
                explanation = new_draft.get("explanation") or ""
                print(f"       ✏️  DRAFT  → گزینه {new_draft.get('answer')}")
                add_row(step_num, node_name.upper(), "REASONER", "assistant",
                        "draft", explanation,
                        meta_answer=new_draft.get("answer"))

            # ── Critic جدید ────────────────────────────────
            if new_critic:
                icon = "🟢" if not new_critic.get("needs_revision") else "🔴"
                print(f"       {icon} CRITIC → needs_revision={new_critic.get('needs_revision')}")
                reasoning = new_critic.get("reasoning") or ""
                add_row(step_num, node_name.upper(), "CRITIC", "assistant",
                        "critic_verdict", reasoning,
                        meta_needs_revision=new_critic.get("needs_revision"),
                        meta_issue=new_critic.get("issue"),
                        meta_action=new_critic.get("action"),
                        meta_reasoning=reasoning)

            # ── Final جدید ─────────────────────────────────
            if new_final:
                explanation = new_final.get("explanation") or ""
                print(f"       🏁 FINAL  → گزینه {new_final.get('answer')}")
                add_row(step_num, node_name.upper(), "FINALIZE", "system",
                        "final", explanation,
                        meta_answer=new_final.get("answer"))

        # ── نتیجه نهایی ────────────────────────────────────
        final_toon    = full_state.get("final_toon") or {}
        draft_toon    = full_state.get("draft_toon") or {}
        verifier_toon = full_state.get("verifier_toon") or {}

        final_ans    = str(final_toon.get("answer")    or "").strip()
        draft_ans    = str(draft_toon.get("answer")    or "").strip()
        verifier_ans = str(verifier_toon.get("answer") or "").strip()
        revisions    = full_state.get("revision_count", 0)
        total_time   = round(time.time() - t_global_start, 2)
        error_msg    = ""

    except Exception as e:
        final_ans    = ""
        draft_ans    = ""
        verifier_ans = ""
        revisions    = 0
        total_time   = round(time.time() - t_global_start, 2)
        error_msg    = str(e)
        print(f"  ❌ خطا: {error_msg[:120]}")
        traceback.print_exc()

    # ── خلاصه SUMMARY در CSV ───────────────────────────────
    v_match = "✅ موافق" if verifier_ans == final_ans else "⚠️ مخالف"
    d_match = "✅ موافق" if draft_ans    == final_ans else "⚠️ مخالف"
    add_row(0, "SUMMARY", "SYSTEM", "summary", "decision_summary",
            f"verifier={verifier_ans} | draft={draft_ans} | final={final_ans}"
            f" | revisions={revisions} | total_time={total_time}s",
            meta_answer=str(final_ans),
            meta_reasoning=f"v_match={v_match} d_match={d_match} error={error_msg}")

    print(f"  ─── نتیجه: final={final_ans} | verifier={verifier_ans} "
          f"| draft={draft_ans} | rev={revisions} | {total_time}s")

    result = {
        "question_number": q_num,
        "final_answer":    final_ans,
        "draft_answer":    draft_ans,
        "verifier_answer": verifier_ans,
        "revision_count":  revisions,
        "total_time_s":    total_time,
        "error":           error_msg,
    }
    return result, conv_rows


print("✅ توابع کمکی تعریف شدند")

✅ توابع کمکی تعریف شدند


In [6]:
# ── تنظیمات ───────────────────────────────────────────────
USE_OPTION_VERIFIER = True
USE_RETRIEVER_TOOL  = False
MAX_REVISIONS       = 2

# ── پوشه خروجی ────────────────────────────────────────────
OUTPUT_DIR = NOTEBOOK_DIR / "eval_results"
OUTPUT_DIR.mkdir(exist_ok=True)

timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
EXCEL_PATH = OUTPUT_DIR / f"eval_403_{timestamp}.xlsx"

# ── تابع مقایسه پاسخ — چند جوابه هم پشتیبانی می‌شود ──────
def is_correct(predicted: str, gold: str) -> bool:
    """
    gold ممکن است "3" یا "1,2" باشد.
    predicted ممکن است "3" یا "1" یا "1,2" باشد.
    """
    pred_set = set(p.strip() for p in str(predicted).split(",") if p.strip())
    gold_set = set(g.strip() for g in str(gold).split(",")      if g.strip())
    return pred_set == gold_set

# ── containers ────────────────────────────────────────────
all_results   = []
all_conv_rows = {}

print(f"🎯 شروع ارزیابی {len(df)} سؤال")
print(f"   use_option_verifier: {USE_OPTION_VERIFIER}")
print(f"   use_retriever_tool : {USE_RETRIEVER_TOOL}")
print(f"   max_revisions      : {MAX_REVISIONS}")
print(f"   خروجی Excel        : {EXCEL_PATH}\n")

batch_start = time.time()

for _, row in df.iterrows():
    q_num    = int(row["question_number"])
    question = str(row["question"])
    options  = str(row["options"])
    gold     = str(row["Gold"]).strip()

    result, conv_rows = run_one_question(
        q_num               = q_num,
        question            = question,
        options_text        = options,
        use_option_verifier = USE_OPTION_VERIFIER,
        use_retriever_tool  = USE_RETRIEVER_TOOL,
        max_revisions       = MAX_REVISIONS,
    )

    result["Gold"]       = gold
    result["correct"]    = is_correct(result["final_answer"], gold)  # ✅ چند جوابه
    result["question"]   = question[:80] + "..."
    result["options"]    = options[:120] + "..."
    result["gold_explain"] = str(row.get("explain") or "")[:300]

    all_results.append(result)
    all_conv_rows[q_num] = conv_rows

    icon = "✅" if result["correct"] else "❌"
    multi = " 🔀" if "," in gold else ""
    print(f"  {icon}  Q{q_num:>3} | final={result['final_answer']} | gold={gold}{multi} | {result['total_time_s']}s\n")

batch_total = round(time.time() - batch_start, 2)
correct_count = sum(1 for r in all_results if r["correct"])
print(f"\n{'═'*65}")
print(f"🏁 ارزیابی تمام شد — زمان کل: {batch_total}s")
print(f"   دقت کلی: {correct_count}/{len(all_results)} = {correct_count/len(all_results)*100:.1f}%")
print(f"{'═'*65}")

🎯 شروع ارزیابی 120 سؤال
   use_option_verifier: True
   use_retriever_tool : False
   max_revisions      : 2
   خروجی Excel        : F:\Thesis\project\3-Multi-Agent-System\Test_402\MA\eval_results\eval_403_20260627_212528.xlsx


═════════════════════════════════════════════════════════════════
🚀 سؤال  1  |  کدام یک از موارد زیر صحیح است؟...
═════════════════════════════════════════════════════════════════
  📍 Step  1 [INITIALIZE  ] next=             msgs= 0 | 0.0s
  📍 Step  2 [SUPERVISOR  ] next=researcher   msgs= 1 | 0.01s
  📍 Step  3 [RESEARCHER  ] next=researcher   msgs= 2 | 10.89s
  📍 Step  4 [SUPERVISOR  ] next=reasoner     msgs= 3 | 10.89s
  📍 Step  5 [REASONER    ] next=reasoner     msgs= 4 | 10.89s
  📍 Step  6 [SUPERVISOR  ] next=tools        msgs= 5 | 10.89s
  📍 Step  7 [TOOLS       ] next=tools        msgs= 6 | 16.06s
  📍 Step  8 [SUPERVISOR  ] next=reasoner     msgs= 6 | 16.06s
  📍 Step  9 [REASONER    ] next=reasoner     msgs= 7 | 23.78s
       ✏️  DRAFT  → گزینه 3
  📍 Step

In [7]:
df_results = pd.DataFrame(all_results)

correct  = df_results["correct"].sum()
total    = len(df_results)
accuracy = correct / total * 100

# سؤالات چند جوابه
multi_q = df_results[df_results["Gold"].str.contains(",", na=False)]
multi_correct = multi_q["correct"].sum()

print(f"\n{'━'*65}")
print(f"  📊 نتایج ارزیابی — مدل Legal Multi-Agent")
print(f"{'━'*65}")
print(f"  Accuracy کل   : {correct}/{total}  =  {accuracy:.1f}%")
print(f"  تک‌جوابه       : {correct - multi_correct}/{total - len(multi_q)}")
print(f"  چند جوابه      : {multi_correct}/{len(multi_q)}")
print(f"  زمان میانگین  : {df_results['total_time_s'].mean():.1f}s / سؤال")
print(f"  revision میانگین: {df_results['revision_count'].mean():.2f}")
print(f"{'━'*65}")

print("\n📋 جزئیات هر سؤال:")
for _, r in df_results.iterrows():
    icon  = "✅" if r["correct"] else "❌"
    multi = " 🔀" if "," in str(r["Gold"]) else ""
    err   = f" [ERR:{r['error'][:40]}]" if r["error"] else ""
    print(f"  {icon} Q{int(r['question_number']):>3} | final={r['final_answer']} | gold={r['Gold']}{multi}"
          f" | verifier={r['verifier_answer']} | rev={r['revision_count']} | {r['total_time_s']}s{err}")

print(f"\n✅ نتایج در df_results ذخیره شد")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  📊 نتایج ارزیابی — مدل Legal Multi-Agent
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Accuracy کل   : 91/120  =  75.8%
  تک‌جوابه       : 91/109
  چند جوابه      : 0/11
  زمان میانگین  : 35.1s / سؤال
  revision میانگین: 0.04
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📋 جزئیات هر سؤال:
  ✅ Q  1 | final=3 | gold=3 | verifier= | rev=0 | 29.91s
  ✅ Q  2 | final=2 | gold=2 | verifier= | rev=0 | 30.72s
  ❌ Q  3 | final=4 | gold=2,4 🔀 | verifier= | rev=0 | 28.6s
  ❌ Q  4 | final=1 | gold=1,4 🔀 | verifier= | rev=2 | 64.19s
  ✅ Q  5 | final=3 | gold=3 | verifier= | rev=0 | 28.83s
  ✅ Q  6 | final=3 | gold=3 | verifier= | rev=0 | 28.97s
  ❌ Q  7 | final=4 | gold=1 | verifier= | rev=0 | 28.25s
  ✅ Q  8 | final=4 | gold=4 | verifier= | rev=0 | 31.65s
  ❌ Q  9 | final=2 | gold=1,2 🔀 | verifier= | rev=0 | 30.69s
  ✅ Q 10 | final=3 | gold=3 | verifier= | rev=0 | 28.32s
  ❌ Q 11 | final=1

In [9]:
import io
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from collections import Counter
from openpyxl.styles import Alignment, PatternFill, Font as XLFont, Border, Side
from openpyxl.drawing.image import Image as XLImage

print(f"💾 در حال ذخیره Excel: {EXCEL_PATH}")

# ══════════════════════════════════════════════════════════════════════════════
# helpers
# ══════════════════════════════════════════════════════════════════════════════
def chart_to_image(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    buf.seek(0)
    img = XLImage(buf)
    plt.close(fig)
    return img

def style_header(ws, hex_color="2E4053"):
    fill = PatternFill("solid", fgColor=hex_color)
    font = XLFont(bold=True, color="FFFFFF", size=11)
    for cell in ws[1]:
        cell.fill = fill
        cell.font = font
        cell.alignment = Alignment(horizontal="center", vertical="center",
                                   wrap_text=True)

def zebra_rows(ws, ca="F2F3F4", cb="FDFEFE"):
    for i, row in enumerate(ws.iter_rows(min_row=2), 2):
        fill = PatternFill("solid", fgColor=ca if i % 2 == 0 else cb)
        for cell in row:
            if cell.fill.fgColor.rgb in ("00000000","FFFFFFFF","00FFFFFF"):
                cell.fill = fill

def add_title(ws, text, row=1, col="A", size=14, color="1A5276"):
    ws[f"{col}{row}"].value = text
    ws[f"{col}{row}"].font  = XLFont(bold=True, size=size, color=color)

BG = "#FAFAFA"
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.facecolor": BG, "axes.facecolor": BG,
})

# ══════════════════════════════════════════════════════════════════════════════
# محاسبات پایه
# ══════════════════════════════════════════════════════════════════════════════
correct_count  = df_results["correct"].sum()
total_count    = len(df_results)
accuracy_val   = round(correct_count / total_count * 100, 2)
multi_mask     = df_results["Gold"].astype(str).str.contains(",", na=False)
multi_total    = int(multi_mask.sum())
multi_correct  = int(df_results.loc[multi_mask, "correct"].sum())
single_total   = int(total_count - multi_total)
single_correct = int(correct_count - multi_correct)

# ── merge با category از df_res_v2 (اگر موجود باشد) ──────────────────────
if "df_res_v2" in dir() and "category" in df_res_v2.columns:
    df_merged = df_results.merge(
        df_res_v2[["question_number","category"]].drop_duplicates(),
        on="question_number", how="left"
    )
    df_merged["category"] = df_merged["category"].fillna("نامشخص")
    has_category = True
else:
    # اگر category مستقیم در df_results نبود، ستون خالی بساز
    df_merged = df_results.copy()
    df_merged["category"] = df_merged.get("category", "نامشخص")
    has_category = "category" in df_results.columns

# ── آمار به تفکیک category ────────────────────────────────────────────────
cat_stats = df_merged.groupby("category").agg(
    total     = ("correct", "count"),
    correct   = ("correct", "sum"),
    avg_time  = ("total_time_s", "mean"),
    avg_rev   = ("revision_count", "mean"),
).reset_index()
cat_stats["accuracy_%"]  = (cat_stats["correct"] / cat_stats["total"] * 100).round(2)
cat_stats["wrong"]       = cat_stats["total"] - cat_stats["correct"]
cat_stats["avg_time"]    = cat_stats["avg_time"].round(2)
cat_stats["avg_rev"]     = cat_stats["avg_rev"].round(3)
cat_stats = cat_stats.sort_values("accuracy_%", ascending=False)

with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl") as writer:

    # ══ Sheet 1: Summary ══════════════════════════════════════════════════
    summary_cols = [
        "question_number","correct","final_answer","Gold",
        "verifier_answer","draft_answer","revision_count",
        "total_time_s","error","question","gold_explain",
    ]
    df_sum = df_results[summary_cols].copy()
    df_sum["correct"] = df_sum["correct"].map({True:"✅", False:"❌"})
    if has_category:
        df_sum.insert(1, "category", df_merged["category"])
    df_sum.to_excel(writer, sheet_name="📊 Summary", index=False)
    ws = writer.sheets["📊 Summary"]
    style_header(ws)
    zebra_rows(ws)
    widths = [8,16,8,10,8,10,10,8,10,30,50,60] if has_category else \
             [8,8,10,8,10,10,8,10,30,50,60]
    for col_letter, w in zip("ABCDEFGHIJKL", widths):
        ws.column_dimensions[col_letter].width = w

    # ══ Sheet 2: Accuracy Stats ════════════════════════════════════════════
    stats_data = {
        "معیار": [
            "تعداد کل سؤالات","تعداد صحیح","تعداد غلط","Accuracy (%)",
            "─────────────────",
            "سؤالات تک‌جوابه","صحیح تک‌جوابه","Accuracy تک‌جوابه (%)",
            "سؤالات چند جوابه","صحیح چند جوابه","Accuracy چند جوابه (%)",
            "─────────────────",
            "میانگین زمان (s)","میانگین Revision","خطاهای اجرایی",
        ],
        "مقدار": [
            total_count, int(correct_count), int(total_count-correct_count),
            accuracy_val, "",
            single_total, single_correct,
            round(single_correct/single_total*100,2) if single_total else 0,
            multi_total, multi_correct,
            round(multi_correct/multi_total*100,2) if multi_total else 0,
            "",
            round(df_results["total_time_s"].mean(),2),
            round(df_results["revision_count"].mean(),3),
            int(df_results["error"].astype(bool).sum()),
        ]
    }
    pd.DataFrame(stats_data).to_excel(writer, sheet_name="📈 Accuracy", index=False)
    ws2 = writer.sheets["📈 Accuracy"]
    style_header(ws2, "1A5276")
    ws2.column_dimensions["A"].width = 30
    ws2.column_dimensions["B"].width = 20

    # ══ Sheet 3: Category Analysis ════════════════════════════════════════
    cat_export = cat_stats.rename(columns={
        "category":"حوزه","total":"تعداد کل","correct":"صحیح",
        "wrong":"غلط","accuracy_%":"Accuracy (%)","avg_time":"میانگین زمان (s)",
        "avg_rev":"میانگین Revision",
    })
    cat_export.to_excel(writer, sheet_name="🏛️ Category Analysis", index=False)
    ws3 = writer.sheets["🏛️ Category Analysis"]
    style_header(ws3, "145A32")
    zebra_rows(ws3, "D5F5E3", "FDFEFE")
    for col, w in zip("ABCDEFG", [28,10,10,10,14,18,18]):
        ws3.column_dimensions[col].width = w

    # ══ Sheet 4: Error Analysis ════════════════════════════════════════════
    df_err = df_merged[~df_merged["correct"]].copy()
    df_err["question_type"]      = df_err["Gold"].astype(str).apply(
        lambda g: "چند جوابه 🔀" if "," in g else "تک‌جوابه")
    df_err["verifier_helped"]    = (
        df_err["draft_answer"].astype(str) != df_err["final_answer"].astype(str)
    ).map({True:"بله ✅", False:"خیر ❌"})
    df_err["model_count"]        = df_err["final_answer"].astype(str).apply(
        lambda x: len(x.split(",")))
    df_err["gold_count"]         = df_err["Gold"].astype(str).apply(
        lambda x: len(x.split(",")))
    df_err["count_mismatch"]     = (
        df_err["model_count"] != df_err["gold_count"]
    ).map({True:"بله ⚠️", False:"خیر"})
    err_cols = ["question_number","category","question_type","final_answer","Gold",
                "draft_answer","verifier_helped","model_count","gold_count",
                "count_mismatch","revision_count","total_time_s","error","question"]
    df_err[[c for c in err_cols if c in df_err.columns]].to_excel(
        writer, sheet_name="❌ Error Analysis", index=False)
    ws4 = writer.sheets["❌ Error Analysis"]
    style_header(ws4, "922B21")
    zebra_rows(ws4, "FADBD8", "FDFEFE")
    for col, w in zip("ABCDEFGHIJKLMN",
                      [8,20,12,10,8,10,12,8,8,12,8,10,30,50]):
        ws4.column_dimensions[col].width = w

    # ══ Sheet 5: Revision Impact ══════════════════════════════════════════
    rev_stats = df_results.groupby("revision_count").agg(
        total    = ("correct","count"),
        correct  = ("correct","sum"),
        avg_time = ("total_time_s","mean"),
        std_time = ("total_time_s","std"),
    ).reset_index()
    rev_stats["accuracy_%"] = (rev_stats["correct"]/rev_stats["total"]*100).round(2)
    rev_stats["avg_time"]   = rev_stats["avg_time"].round(2)
    rev_stats["std_time"]   = rev_stats["std_time"].fillna(0).round(2)
    rev_stats.columns = ["تعداد Revision","تعداد کل","تعداد صحیح",
                         "میانگین زمان (s)","انحراف زمان","Accuracy (%)"]
    rev_stats.to_excel(writer, sheet_name="🔄 Revision Impact", index=False)
    ws5 = writer.sheets["🔄 Revision Impact"]
    style_header(ws5, "1F618D")
    zebra_rows(ws5, "D6EAF8", "FDFEFE")
    for col, w in zip("ABCDEF", [18,12,12,18,14,14]):
        ws5.column_dimensions[col].width = w

    # ══ Sheet 6: Verifier Effect ══════════════════════════════════════════
    df_v = df_results.copy()
    df_v["verifier_changed"] = (
        df_v["draft_answer"].astype(str) != df_v["final_answer"].astype(str))
    df_v["change_direction"] = df_v.apply(lambda r: (
        "بهبود ✅"                if r["verifier_changed"] and r["correct"] else
        "تخریب ❌"                if r["verifier_changed"] and not r["correct"] else
        "بدون تغییر — صحیح ✅" if not r["verifier_changed"] and r["correct"] else
        "بدون تغییر — غلط ❌"
    ), axis=1)
    verifier_stats = df_v.groupby("change_direction").agg(
        count=("correct","count")).reset_index()
    verifier_stats["درصد"] = (verifier_stats["count"]/total_count*100).round(2)
    verifier_stats.columns = ["وضعیت Verifier","تعداد","درصد (%)"]
    verifier_stats.to_excel(writer, sheet_name="🔍 Verifier Effect", index=False)
    ws6 = writer.sheets["🔍 Verifier Effect"]
    style_header(ws6, "117A65")
    zebra_rows(ws6, "D5F5E3", "FDFEFE")
    for col, w in zip("ABC", [30,10,12]):
        ws6.column_dimensions[col].width = w

    # ══ Sheet 7: Time Analysis ════════════════════════════════════════════
    df_t = df_results.copy()
    df_t["time_bucket"] = pd.cut(
        df_t["total_time_s"],
        bins=[0,10,20,30,45,60,9999],
        labels=["<10s","10-20s","20-30s","30-45s","45-60s",">60s"])
    time_stats = df_t.groupby("time_bucket", observed=True).agg(
        count  = ("correct","count"),
        correct = ("correct","sum"),
    ).reset_index()
    time_stats["accuracy_%"] = (time_stats["correct"]/time_stats["count"]*100).round(2)
    time_stats.columns = ["بازه زمانی","تعداد","صحیح","Accuracy (%)"]
    corr_val = df_results[["total_time_s","correct"]].corr().iloc[0,1]
    time_extra = pd.DataFrame({
        "معیار": ["همبستگی زمان-صحت","میانگین زمان صحیح","میانگین زمان غلط"],
        "مقدار": [
            round(corr_val,4),
            round(df_results[df_results["correct"]]["total_time_s"].mean(),2),
            round(df_results[~df_results["correct"]]["total_time_s"].mean(),2),
        ]
    })
    time_stats.to_excel(writer, sheet_name="⏱️ Time Analysis", index=False)
    time_extra.to_excel(writer, sheet_name="⏱️ Time Analysis", index=False,
                        startrow=len(time_stats)+3)
    ws7 = writer.sheets["⏱️ Time Analysis"]
    style_header(ws7, "6C3483")
    zebra_rows(ws7, "E8DAEF", "FDFEFE")
    for col, w in zip("ABCD", [14,10,10,14]):
        ws7.column_dimensions[col].width = w

    # ══ Sheet 8: Answer Distribution ═════════════════════════════════════
    def extract_options(series):
        opts = []
        for val in series.astype(str):
            for o in val.split(","):
                o = o.strip()
                if o: opts.append(o)
        return Counter(opts)

    model_cnt = extract_options(df_results["final_answer"])
    gold_cnt  = extract_options(df_results["Gold"])
    all_opts  = sorted(set(list(model_cnt.keys())+list(gold_cnt.keys())))
    dist_df   = pd.DataFrame({
        "گزینه":             all_opts,
        "انتخاب مدل":        [model_cnt.get(o,0) for o in all_opts],
        "پاسخ صحیح (Gold)": [gold_cnt.get(o,0)  for o in all_opts],
    })
    dist_df["اختلاف"] = dist_df["انتخاب مدل"] - dist_df["پاسخ صحیح (Gold)"]
    dist_df.to_excel(writer, sheet_name="📉 Answer Distribution", index=False)
    ws8 = writer.sheets["📉 Answer Distribution"]
    style_header(ws8, "784212")
    zebra_rows(ws8, "FDEBD0", "FDFEFE")
    for col, w in zip("ABCD", [10,14,18,10]):
        ws8.column_dimensions[col].width = w

    # ══ Sheet 9: Charts ═══════════════════════════════════════════════════
    pd.DataFrame().to_excel(writer, sheet_name="📊 Charts", index=False)
    wsc = writer.sheets["📊 Charts"]
    add_title(wsc, "📊 نمودارهای تحلیلی — Legal Multi-Agent Evaluation")

    # ── نمودار 1: Pie — نرخ کلی دقت ──────────────────────────────────────
    fig1, ax1 = plt.subplots(figsize=(5,4), facecolor=BG)
    ax1.pie([correct_count, total_count-correct_count],
            labels=["صحیح","غلط"], colors=["#27AE60","#E74C3C"],
            autopct="%1.1f%%", startangle=90,
            wedgeprops=dict(edgecolor="white", linewidth=2))
    ax1.set_title(f"نرخ کلی دقت  ({accuracy_val}%)", fontsize=13, fontweight="bold")
    wsc.add_image(chart_to_image(fig1), "B3")

    # ── نمودار 2: Bar — تک‌جوابه vs چند جوابه ────────────────────────────
    fig2, ax2 = plt.subplots(figsize=(5,4), facecolor=BG)
    accs2 = [round(single_correct/single_total*100,1) if single_total else 0,
             round(multi_correct/multi_total*100,1)   if multi_total  else 0]
    bars2 = ax2.bar(["تک‌جوابه","چند جوابه"], accs2,
                    color=["#2980B9","#8E44AD"], width=0.5, edgecolor="white")
    for b,v in zip(bars2, accs2):
        ax2.text(b.get_x()+b.get_width()/2, v+1, f"{v}%",
                 ha="center", fontweight="bold")
    ax2.set_ylim(0,110); ax2.set_ylabel("Accuracy (%)")
    ax2.set_title("تک‌جوابه vs چند جوابه", fontsize=13, fontweight="bold")
    wsc.add_image(chart_to_image(fig2), "J3")

    # ── نمودار 3: Horizontal Bar — دقت به تفکیک حوزه ─────────────────────
    fig3, ax3 = plt.subplots(figsize=(8, max(4, len(cat_stats)*0.55)), facecolor=BG)
    cats_sorted = cat_stats.sort_values("accuracy_%")
    colors3 = ["#27AE60" if v >= 70 else "#F39C12" if v >= 50 else "#E74C3C"
               for v in cats_sorted["accuracy_%"]]
    bars3 = ax3.barh(cats_sorted["category"], cats_sorted["accuracy_%"],
                     color=colors3, edgecolor="white", height=0.6)
    for bar, val in zip(bars3, cats_sorted["accuracy_%"]):
        ax3.text(val+0.5, bar.get_y()+bar.get_height()/2,
                 f"{val:.1f}%", va="center", fontsize=9, fontweight="bold")
    ax3.set_xlim(0, 115)
    ax3.axvline(accuracy_val, color="#2C3E50", linestyle="--",
                linewidth=1.5, label=f"میانگین کل: {accuracy_val}%")
    ax3.legend(fontsize=9)
    ax3.set_xlabel("Accuracy (%)")
    ax3.set_title("دقت مدل به تفکیک حوزه حقوقی", fontsize=13, fontweight="bold")
    fig3.tight_layout()
    wsc.add_image(chart_to_image(fig3), "B25")

    # ── نمودار 4: Grouped Bar — صحیح/غلط به تفکیک حوزه ──────────────────
    fig4, ax4 = plt.subplots(figsize=(8, max(4, len(cat_stats)*0.55)), facecolor=BG)
    y4  = np.arange(len(cat_stats))
    h4  = 0.35
    cs4 = cat_stats.sort_values("total", ascending=True)
    ax4.barh(y4 - h4/2, cs4["correct"], height=h4,
             color="#27AE60", label="صحیح", edgecolor="white")
    ax4.barh(y4 + h4/2, cs4["wrong"],   height=h4,
             color="#E74C3C", label="غلط",  edgecolor="white")
    ax4.set_yticks(y4)
    ax4.set_yticklabels(cs4["category"], fontsize=9)
    ax4.set_xlabel("تعداد")
    ax4.set_title("صحیح / غلط به تفکیک حوزه", fontsize=13, fontweight="bold")
    ax4.legend()
    fig4.tight_layout()
    wsc.add_image(chart_to_image(fig4), "R25")

    # ── نمودار 5: Scatter — دقت vs تعداد سؤال (اندازه = زمان) ────────────
    fig5, ax5 = plt.subplots(figsize=(7,5), facecolor=BG)
    sc5 = ax5.scatter(
        cat_stats["total"],
        cat_stats["accuracy_%"],
        s=cat_stats["avg_time"]*10,
        c=cat_stats["accuracy_%"],
        cmap="RdYlGn", vmin=0, vmax=100,
        alpha=0.8, edgecolors="#2C3E50", linewidth=0.8)
    for _, r in cat_stats.iterrows():
        ax5.annotate(r["category"], (r["total"], r["accuracy_%"]),
                     fontsize=8, ha="center", va="bottom",
                     xytext=(0, 6), textcoords="offset points")
    plt.colorbar(sc5, ax=ax5, label="Accuracy (%)")
    ax5.set_xlabel("تعداد سؤالات")
    ax5.set_ylabel("Accuracy (%)")
    ax5.set_title("دقت vs حجم سؤالات\n(اندازه دایره = میانگین زمان)",
                  fontsize=12, fontweight="bold")
    wsc.add_image(chart_to_image(fig5), "B55")

    # ── نمودار 6: Bar — تأثیر Revision ───────────────────────────────────
    rev_g = df_results.groupby("revision_count")["correct"].mean().reset_index()
    fig6, ax6 = plt.subplots(figsize=(5,4), facecolor=BG)
    bars6 = ax6.bar(rev_g["revision_count"].astype(str),
                    rev_g["correct"]*100, color="#1ABC9C",
                    width=0.5, edgecolor="white")
    for b,v in zip(bars6, rev_g["correct"]*100):
        ax6.text(b.get_x()+b.get_width()/2, v+1,
                 f"{v:.1f}%", ha="center", fontweight="bold")
    ax6.set_ylim(0,110)
    ax6.set_xlabel("تعداد Revision"); ax6.set_ylabel("Accuracy (%)")
    ax6.set_title("تأثیر Revision بر دقت", fontsize=13, fontweight="bold")
    wsc.add_image(chart_to_image(fig6), "J55")

    # ── نمودار 7: Verifier Effect Stacked ────────────────────────────────
    vc = df_v["change_direction"].value_counts()
    fig7, ax7 = plt.subplots(figsize=(6,4), facecolor=BG)
    cmap7 = {"بهبود ✅":"#27AE60","تخریب ❌":"#E74C3C",
             "بدون تغییر — صحیح ✅":"#2ECC71","بدون تغییر — غلط ❌":"#C0392B"}
    ax7.bar(range(len(vc)), vc.values,
            color=[cmap7.get(k,"#7F8C8D") for k in vc.index], edgecolor="white")
    ax7.set_xticks(range(len(vc)))
    ax7.set_xticklabels(vc.index, rotation=20, ha="right", fontsize=9)
    ax7.set_ylabel("تعداد")
    ax7.set_title("وضعیت Option Verifier", fontsize=13, fontweight="bold")
    for i,v in enumerate(vc.values):
        ax7.text(i, v+0.3, str(v), ha="center", fontweight="bold")
    wsc.add_image(chart_to_image(fig7), "B75")

    # ── نمودار 8: Scatter — زمان vs Revision ─────────────────────────────
    fig8, ax8 = plt.subplots(figsize=(6,4), facecolor=BG)
    clrs8 = ["#27AE60" if c else "#E74C3C" for c in df_results["correct"]]
    ax8.scatter(df_results["total_time_s"], df_results["revision_count"],
                c=clrs8, alpha=0.7, s=60, edgecolors="white", linewidth=0.5)
    ax8.set_xlabel("زمان (s)"); ax8.set_ylabel("تعداد Revision")
    ax8.set_title("زمان در برابر Revision", fontsize=13, fontweight="bold")
    ax8.legend(handles=[mpatches.Patch(color="#27AE60", label="صحیح"),
                        mpatches.Patch(color="#E74C3C", label="غلط")])
    wsc.add_image(chart_to_image(fig8), "J75")

    # ── نمودار 9: Grouped Bar — توزیع گزینه‌ها ───────────────────────────
    fig9, ax9 = plt.subplots(figsize=(6,4), facecolor=BG)
    x9 = np.arange(len(dist_df)); w9 = 0.35
    ax9.bar(x9-w9/2, dist_df["انتخاب مدل"],
            width=w9, label="مدل", color="#3498DB", edgecolor="white")
    ax9.bar(x9+w9/2, dist_df["پاسخ صحیح (Gold)"],
            width=w9, label="Gold", color="#E67E22", edgecolor="white")
    ax9.set_xticks(x9); ax9.set_xticklabels(dist_df["گزینه"])
    ax9.set_ylabel("فراوانی")
    ax9.set_title("توزیع گزینه: مدل vs Gold", fontsize=13, fontweight="bold")
    ax9.legend()
    wsc.add_image(chart_to_image(fig9), "B92")

    # ══ Sheet 10+: مکالمه هر سؤال ════════════════════════════════════════
    for q_num, rows in all_conv_rows.items():
        sname = f"Q{q_num:03d}_conv"
        if not rows:
            pd.DataFrame(columns=CONV_COLUMNS).to_excel(
                writer, sheet_name=sname, index=False)
            continue
        df_conv = pd.DataFrame(rows, columns=CONV_COLUMNS)
        df_conv.to_excel(writer, sheet_name=sname, index=False)
        ws_q = writer.sheets[sname]
        style_header(ws_q, "4A235A")
        for col, w in zip("ABCDEFGHIJK",
                          [6,14,14,10,18,80,10,16,30,30,60]):
            ws_q.column_dimensions[col].width = w
        for row_cells in ws_q.iter_rows(min_row=2):
            for cell in row_cells:
                cell.alignment = Alignment(wrap_text=True, vertical="top")
        print(f"  ✅ sheet {sname:15} → {len(df_conv)} ردیف")

# ══ خروجی کنسول df_res_v2 ════════════════════════════════════════════════════
if "df_res_v2" in dir():
    overall_acc = df_res_v2["is_correct"].mean() if "is_correct" in df_res_v2 else None
    coverage    = df_res_v2["pred"].notna().mean() if "pred" in df_res_v2 else None
    error_rate  = df_res_v2["error"].notna().mean() if "error" in df_res_v2 else None
    print(f"\nOverall accuracy (v2): {overall_acc}")
    print(f"Coverage (has pred)  : {coverage}")
    print(f"Error rate           : {error_rate}")
    if "category" in df_res_v2.columns and "is_correct" in df_res_v2.columns:
        print("\n── دقت به تفکیک حوزه (df_res_v2) ──")
        print(df_res_v2.groupby("category")["is_correct"].mean()
                       .sort_values(ascending=False).to_string())
    out_path = PROJECT_ROOT / "data" / "eval" / "qwen3-80.csv"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df_res_v2.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"\n✅ Saved to: {out_path}")

print(f"""
🎉 Excel ذخیره شد: {EXCEL_PATH}
   📊 Summary              خلاصه همه سؤالات + category
   📈 Accuracy             آمار کلی + تک/چند جوابه
   🏛️  Category Analysis   دقت به تفکیک حوزه
   ❌ Error Analysis        تحلیل خطاها
   🔄 Revision Impact       تأثیر revision
   🔍 Verifier Effect       تأثیر option verifier
   ⏱️  Time Analysis        تحلیل زمان
   📉 Answer Distribution   توزیع گزینه‌ها
   📊 Charts                9 نمودار تعبیه‌شده
   Q###_conv × {len(all_conv_rows)}
""")

💾 در حال ذخیره Excel: F:\Thesis\project\3-Multi-Agent-System\Test_402\MA\eval_results\eval_403_20260627_212528.xlsx


C:\Users\sazgar\AppData\Local\Temp\ipykernel_28920\3455218214.py:18: UserWarning: Glyph 9989 (\N{WHITE HEAVY CHECK MARK}) missing from font(s) DejaVu Sans.
  fig.savefig(buf, format="png", dpi=150, bbox_inches="tight",
C:\Users\sazgar\AppData\Local\Temp\ipykernel_28920\3455218214.py:18: UserWarning: Glyph 10060 (\N{CROSS MARK}) missing from font(s) DejaVu Sans.
  fig.savefig(buf, format="png", dpi=150, bbox_inches="tight",


  ✅ sheet Q001_conv       → 35 ردیف
  ✅ sheet Q002_conv       → 35 ردیف
  ✅ sheet Q003_conv       → 35 ردیف
  ✅ sheet Q004_conv       → 53 ردیف
  ✅ sheet Q005_conv       → 35 ردیف
  ✅ sheet Q006_conv       → 35 ردیف
  ✅ sheet Q007_conv       → 35 ردیف
  ✅ sheet Q008_conv       → 35 ردیف
  ✅ sheet Q009_conv       → 35 ردیف
  ✅ sheet Q010_conv       → 35 ردیف
  ✅ sheet Q011_conv       → 35 ردیف
  ✅ sheet Q012_conv       → 35 ردیف
  ✅ sheet Q013_conv       → 35 ردیف
  ✅ sheet Q014_conv       → 35 ردیف
  ✅ sheet Q015_conv       → 35 ردیف
  ✅ sheet Q016_conv       → 35 ردیف
  ✅ sheet Q017_conv       → 35 ردیف
  ✅ sheet Q018_conv       → 35 ردیف
  ✅ sheet Q019_conv       → 35 ردیف
  ✅ sheet Q020_conv       → 35 ردیف
  ✅ sheet Q021_conv       → 35 ردیف
  ✅ sheet Q022_conv       → 35 ردیف
  ✅ sheet Q023_conv       → 35 ردیف
  ✅ sheet Q024_conv       → 35 ردیف
  ✅ sheet Q025_conv       → 35 ردیف
  ✅ sheet Q026_conv       → 35 ردیف
  ✅ sheet Q027_conv       → 35 ردیف
  ✅ sheet Q028_conv       → 